# Train VGG16 - Dog Breed Classification
Notebook đơn giản để train VGG16 trên Colab

## 1. Setup: Clone Repository

In [ ]:
# Thay đổi thông tin repo của bạn
REPO_URL = "https://github.com/OWNER/REPO_NAME.git"
BRANCH_NAME = "vinhkhoa"
REPO_NAME = "Project_DBM"

!git clone {REPO_URL}
%cd {REPO_NAME}
!git checkout {BRANCH_NAME}
!git status

## 2. Kiểm tra GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not available! Go to Runtime > Change runtime type > Select GPU")

## 3. Cài đặt Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Mount Google Drive (để lấy dataset)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy dataset từ Drive (thay đổi path nếu cần)
# !cp -r /content/drive/MyDrive/dog-breed-dataset/train ./train
# !cp /content/drive/MyDrive/dog-breed-dataset/labels.csv ./labels.csv

## 5. Kiểm tra Dataset

In [ ]:
import os
import pandas as pd

if os.path.exists('labels.csv'):
    df = pd.read_csv('labels.csv')
    print(f"✓ Total samples: {len(df)}")
    print(f"✓ Number of breeds: {df['breed'].nunique()}")
else:
    print("❌ labels.csv not found! Please upload dataset.")

if os.path.exists('train'):
    print(f"✓ Training images: {len(os.listdir('train'))}")
else:
    print("❌ train directory not found! Please upload dataset.")

## 6. 📊 DATA MINING & EXPLORATORY DATA ANALYSIS

Phân tích dữ liệu trước khi train:
- Dataset statistics
- Class imbalance analysis  
- Image properties
- Data quality check
- Visualizations & insights

In [ ]:
!python data_mining_analysis.py

### View Data Mining Results

In [ ]:
import json
from IPython.display import Image, display

with open('data_mining_results/mining_report.json', 'r') as f:
    report = json.load(f)

print("="*60)
print("  DATA MINING REPORT")
print("="*60)
print(f"\nDataset: {report['dataset_overview']['total_samples']} samples, {report['dataset_overview']['num_classes']} classes")
print(f"Imbalance ratio: {report['class_imbalance']['imbalance_ratio']:.2f}")
print(f"\nKey Insights:")
for insight in report['insights']:
    print(f"  • {insight}")
print(f"\nRecommendations:")
for rec in report['recommendations']:
    print(f"  • {rec}")
print("="*60)

# Display charts
print("\nClass Distribution:")
display(Image('data_mining_results/class_distribution.png'))
print("\nImage Properties:")
display(Image('data_mining_results/image_properties.png'))

## 7. 🚀 Train VGG16

Output sẽ hiển thị đầy đủ pipeline:
- STEP 1: Dataset Loading
- STEP 2: Preprocessing + Augmentation
- STEP 3: Train/Val/Test Split
- STEP 4: Feature Extraction (VGG16)
- STEP 5: Training (Fine-tune)
- STEP 6: Evaluation (Acc, F1, Precision, Recall)

In [ ]:
!python train_vgg.py

## 8. View Results

In [ ]:
import json

with open('training_models/vgg_test_metrics.json', 'r') as f:
    metrics = json.load(f)

print("="*60)
print("  VGG16 FINAL RESULTS")
print("="*60)
print(f"Accuracy:  {metrics['test_accuracy']:.4f} ({metrics['test_accuracy']*100:.2f}%)")
print(f"Precision: {metrics['test_precision']:.4f}")
print(f"Recall:    {metrics['test_recall']:.4f}")
print(f"F1-Score:  {metrics['test_f1']:.4f}")
print("="*60)

## 9. Plot Training History

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

with open('training_models/vgg_training_history.json', 'r') as f:
    history = json.load(f)

df_history = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training Loss
axes[0].plot(df_history['epoch'], df_history['train_loss'], marker='o', color='#e74c3c', linewidth=2)
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Loss', fontweight='bold')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].grid(alpha=0.3)

# Validation Metrics
axes[1].plot(df_history['epoch'], df_history['val_accuracy'], marker='o', label='Accuracy', linewidth=2)
axes[1].plot(df_history['epoch'], df_history['val_f1'], marker='s', label='F1-Score', linewidth=2)
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Score', fontweight='bold')
axes[1].set_title('Validation Metrics', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tensorboard_vgg

## 11. Test Prediction

In [ ]:
import torch
from model_vgg import DogBreedVGG16
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
checkpoint = torch.load('training_models/best_vgg.pth', map_location=device)
num_classes = len(checkpoint['class_to_idx'])
model = DogBreedVGG16(num_classes=num_classes, pretrained=False).to(device)
model.load_state_dict(checkpoint['model'])
model.eval()

idx_to_class = {idx: class_name for class_name, idx in checkpoint['class_to_idx'].items()}

# Test với một ảnh random
df = pd.read_csv('labels.csv')
sample = df.sample(n=1).iloc[0]
image_path = os.path.join('train', sample['id'] + '.jpg')

transform = Compose([
    Resize((224, 224)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

image = Image.open(image_path).convert('RGB')
image_tensor = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(image_tensor)
    probabilities = torch.nn.functional.softmax(output, dim=1)
    top5_prob, top5_idx = torch.topk(probabilities, 5)

# Display
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.axis('off')
plt.title(f'True: {sample["breed"]}', fontweight='bold')

plt.subplot(1, 2, 2)
breeds = [idx_to_class[idx.item()] for idx in top5_idx[0]]
probs = [prob.item() * 100 for prob in top5_prob[0]]
plt.barh(breeds, probs, color='#3498db')
plt.xlabel('Probability (%)', fontweight='bold')
plt.title('Top 5 Predictions', fontweight='bold')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

print(f"\nPredicted: {breeds[0]} ({probs[0]:.2f}%)")
print(f"True label: {sample['breed']}")
print(f"Correct: {'✓' if breeds[0] == sample['breed'] else '✗'}")

## 12. Download Results

In [ ]:
from google.colab import files

# Download model và metrics
files.download('training_models/best_vgg.pth')
files.download('training_models/vgg_test_metrics.json')

print("✓ Downloaded!")

## 13. Backup to Drive

In [ ]:
# Backup toàn bộ kết quả vào Drive
!mkdir -p /content/drive/MyDrive/dog_breed_vgg
!cp -r training_models /content/drive/MyDrive/dog_breed_vgg/
!cp -r tensorboard_vgg /content/drive/MyDrive/dog_breed_vgg/

print("✓ Backup completed to /content/drive/MyDrive/dog_breed_vgg/")